# SHAP Explainability

This notebook loads the final trained model and adds SHAP values, so every
prediction comes with a clear reason - which features pushed the score up
or down for a specific company.

In [1]:
# Loading the final features table and retraining the model.
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
import xgboost as xgb

features_df = pd.read_parquet("../data/features.parquet")
features_df.shape

(2511, 12)

In [2]:
# Splitting into train and test by snapshot date, matching the modelling notebook.
features_df = features_df.sort_values("snapshot_date")
split_index = int(len(features_df) * 0.8)
split_date = features_df.iloc[split_index]["snapshot_date"]

train = features_df[features_df["snapshot_date"] < split_date]
test = features_df[features_df["snapshot_date"] >= split_date]

print(len(train), len(test), split_date)

2008 503 2025-01-21 00:00:00


In [3]:
# Retraining the final model with all 9 features, for use in SHAP.
feature_cols = [
    "days_since_last_accounts", "count_late_confirmation_statements",
    "count_recent_resignations", "count_new_charges", "company_age_years",
    "longest_filing_gap", "accounts_missing", "filing_gap_missing",
    "director_distress_score_safe",
]

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05)
model.fit(train[feature_cols], train["is_failed"])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [4]:
# Confirming the retrained model matches the established result.
model_scores = model.predict_proba(test[feature_cols])[:, 1]

def precision_at_k(y_true, y_scores, k_percent=10):
    """Return the precision among the top k percent highest scored companies."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    return y_true.iloc[top_k_idx].mean()

precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))

np.float64(0.68)

## Feature importance and per-company explanations

In [5]:
# Explaining the model's predictions with SHAP values.
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(test[feature_cols])
shap_values.shape

(503, 9)

In [6]:
# Summarising average feature importance across all test companies.
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)
importance

company_age_years                     0.612517
filing_gap_missing                    0.292701
longest_filing_gap                    0.238080
director_distress_score_safe          0.197959
days_since_last_accounts              0.160730
count_recent_resignations             0.080758
count_late_confirmation_statements    0.026763
count_new_charges                     0.024113
accounts_missing                      0.000000
dtype: float32

In [7]:
# Checking whether company age differs systematically between failed and live companies.
features_df.groupby("is_failed")["company_age_years"].describe()

,count,mean,std,min,25%,50%,75%,max
is_failed,,,,,,,,
0,1266.0,8.039640,10.970993,0.002738,1.062971,4.117728,10.784394,146.715948
1,1245.0,11.644101,13.775721,0.016427,4.065708,7.564682,13.245722,119.523614


In [8]:
# Looking at the SHAP explanation for one real company in the test set.
company_index = 0
company_number = test.iloc[company_index]["CompanyNumber"]
company_shap = shap_values[company_index]

explanation = pd.Series(company_shap, index=feature_cols).sort_values(key=abs, ascending=False)
print("Company:", company_number)
print("Predicted risk score:", model_scores[company_index])
explanation

Company: 08670145
Predicted risk score: 0.60416937


company_age_years                     0.399564
days_since_last_accounts             -0.149521
filing_gap_missing                    0.097604
director_distress_score_safe         -0.063914
count_recent_resignations            -0.050053
longest_filing_gap                    0.024543
count_late_confirmation_statements    0.015675
count_new_charges                    -0.013271
accounts_missing                      0.000000
dtype: float32

In [9]:
# Turning SHAP values into a plain language explanation for one company.
def explain_prediction(company_shap: np.ndarray, feature_cols: list, top_n: int = 3) -> list[str]:
    """Return the top n features driving a prediction, as plain English sentences."""
    explanation = pd.Series(company_shap, index=feature_cols).sort_values(key=abs, ascending=False)
    sentences = []
    labels = {
        "company_age_years": "the company's age",
        "days_since_last_accounts": "how recently accounts were filed",
        "count_late_confirmation_statements": "late confirmation statements",
        "count_recent_resignations": "recent director resignations",
        "count_new_charges": "new charges registered against the company",
        "longest_filing_gap": "the longest gap between filings",
        "accounts_missing": "whether an accounts due date was on record",
        "filing_gap_missing": "whether the company had enough filing history",
        "director_distress_score_safe": "whether the company's directors have a history of other company failures",
    }
    for feature, value in explanation.head(top_n).items():
        direction = "increased" if value > 0 else "decreased"
        sentences.append(f"{labels.get(feature, feature)} {direction} the risk score")
    return sentences

In [10]:
# Testing the explanation function on the same example company.
explain_prediction(company_shap, feature_cols)

["the company's age increased the risk score",
 'how recently accounts were filed decreased the risk score',
 'whether the company had enough filing history increased the risk score']